# 01b · Tile plan

The DNR portal blocks scripted downloads, so full-AOI LiDAR comes through the browser one tile at a time. This notebook decides **which tiles** to download.

- **Training tiles:** stratified random sample by elevation among forested tiles → Phase 3 Sentinel-2 model.
- **Stream tiles:** every tile touching a 100 m buffer of perennial NHD streams → Phase 5 riparian analysis.
- Output: `figures/01b_tile_plan.csv`, `figures/01b_tile_plan.png`, and clickable download links.

Tile grid is rebuilt from DNR's naming (lower-left corner in WA State Plane South ft, 2250 ft tiles), checked against the 01 test tile.

In [ ]:
REPO_URL  <- "https://github.com/MegaDeadCowboy/forest-height-s2-lidar.git"
REPO_NAME <- "forest-height-s2-lidar"
if (dir.exists("/content") && !dir.exists(file.path("/content", REPO_NAME)))
  system(paste("git clone --depth 1", REPO_URL, file.path("/content", REPO_NAME)))
if (dir.exists("/content")) setwd(file.path("/content", REPO_NAME)) else if (basename(getwd()) == "notebooks") setwd("..")
system("git pull -q")

for (f in c("setup", "config", "aoi", "tiles", "layers")) source(file.path("R", paste0(f, ".R")))
setup_packages(c("sf", "terra", "dplyr", "ggplot2", "readr", "httr2", "rstac", "IRdisplay"))

CASCADES_NORTH_ID <- 1734        # portal dataset id (from the 01 test-tile link)
TEST_TILE         <- "w1172250n125100"
MB_PER_TILE       <- 73.7        # measured on the test tile; tiles vary
N_PER_STRATUM     <- 12          # 4 elevation strata -> 48 training tiles
SEED              <- 20261027
dir.create("figures", showWarnings = FALSE)

## 1 · AOI and tile grid

In [ ]:
a <- load_aoi(AOI_BBOX, pattern = "Whatcom")
tiles <- dnr_tile_grid(a$aoi)
cat(sprintf("Tiles touching AOI: %d  |  test tile in grid: %s\n", nrow(tiles), TEST_TILE %in% tiles$tile))
stopifnot("Test tile not reproduced by the grid — naming rule is wrong" = TEST_TILE %in% tiles$tile)

## 2 · Per-tile attributes
Elevation from USGS 3DEP (30 m), tree cover share from ESA WorldCover (class 10), and perennial stream length from NHD (FType 460, excluding intermittent 46003 and ephemeral 46007 — a proxy until DNR fish-bearing water types in Phase 5).

In [ ]:
dem <- usgs_dem(a$aoi, res = 30)
wc  <- worldcover(a$aoi)
v   <- terra::vect(tiles)

tiles$elev_mean   <- round(terra::extract(dem, v, fun = mean, na.rm = TRUE)[, 2])
tiles$forest_frac <- round(terra::extract(wc == 10, terra::project(v, terra::crs(wc)), fun = mean, na.rm = TRUE)[, 2], 3)
summary(sf::st_drop_geometry(tiles)[, c("aoi_frac", "elev_mean", "forest_frac")])

In [ ]:
fl <- nhd_flowlines(a$aoi)
fl$fcode <- as.integer(fl$fcode); fl$ftype <- as.integer(fl$ftype)
streams <- fl[fl$ftype == 460 & !fl$fcode %in% c(46003L, 46007L), ]
cat(sprintf("NHD flowlines: %d  |  perennial streams kept: %d  (%.1f km)\n",
            nrow(fl), nrow(streams), sum(as.numeric(sf::st_length(streams))) / 1000))
print(table(fl$fcode))

buf <- sf::st_union(sf::st_buffer(streams, 100))
tiles$stream <- lengths(sf::st_intersects(tiles, buf)) > 0

## 3 · Training sample
Eligible: ≥ 50% inside the AOI and ≥ 50% WorldCover tree cover. Four elevation strata (quartiles of eligible tiles), `N_PER_STRATUM` random tiles each, plus the test tile.

In [ ]:
set.seed(SEED)
elig <- tiles$aoi_frac >= 0.5 & tiles$forest_frac >= 0.5 & !is.na(tiles$elev_mean)
brks <- quantile(tiles$elev_mean[elig], c(0, .25, .5, .75, 1))
tiles$elev_band <- cut(tiles$elev_mean, brks, include.lowest = TRUE, dig.lab = 4)

train_ids <- sf::st_drop_geometry(tiles[elig, ]) |>
  dplyr::group_by(elev_band) |>
  dplyr::slice_sample(n = N_PER_STRATUM) |>
  dplyr::pull(tile)
tiles$train <- tiles$tile %in% c(train_ids, TEST_TILE)

tiles$role <- dplyr::case_when(tiles$train & tiles$stream ~ "train + stream",
                               tiles$train ~ "train", tiles$stream ~ "stream", TRUE ~ "not needed")
plan <- tiles[tiles$role != "not needed", ]
plan$downloaded <- plan$tile == TEST_TILE
plan$url <- dnr_portal_url(plan, CASCADES_NORTH_ID)

tab <- sf::st_drop_geometry(tiles) |> dplyr::count(role)
print(tab)
print(table(train = tiles$train, band = tiles$elev_band, useNA = "no"))
cat(sprintf("\nTo download: %d tiles  ≈ %.1f GB (at %.1f MB/tile)\n",
            sum(!plan$downloaded), sum(!plan$downloaded) * MB_PER_TILE / 1000, MB_PER_TILE))

In [ ]:
ctr <- sf::st_coordinates(sf::st_transform(sf::st_centroid(sf::st_geometry(plan)), 4326))
out <- sf::st_drop_geometry(plan) |>
  dplyr::mutate(lon = round(ctr[, 1], 5), lat = round(ctr[, 2], 5), elev_band = as.character(elev_band)) |>
  dplyr::arrange(dplyr::desc(train), dplyr::desc(stream), tile) |>
  dplyr::mutate(n = dplyr::row_number()) |>
  dplyr::select(n, tile, role, elev_mean, elev_band, forest_frac, aoi_frac, lon, lat, downloaded, url)
readr::write_csv(out, "figures/01b_tile_plan.csv")
head(out[, 1:9], 10)

## 4 · Map

In [ ]:
p <- ggplot2::ggplot() +
  ggplot2::geom_sf(data = tiles, fill = "grey95", colour = "grey80", linewidth = 0.1) +
  ggplot2::geom_sf(data = plan, ggplot2::aes(fill = role), colour = "white", linewidth = 0.1) +
  ggplot2::geom_sf(data = streams, colour = "#1f78b4", linewidth = 0.3) +
  ggplot2::geom_sf(data = a$aoi, fill = NA, colour = "black", linewidth = 0.5) +
  ggplot2::scale_fill_manual(values = c("train" = "#e6550d", "train + stream" = "#756bb1", "stream" = "#9ecae1")) +
  ggplot2::coord_sf(crs = CRS_EPSG) +
  ggplot2::labs(title = "LiDAR tile plan, Lake Whatcom AOI", fill = NULL,
                caption = sprintf("DNR 2250 ft tiles · %d training, %d stream · NHD perennial streams · EPSG:%d",
                                  sum(plan$train), sum(plan$stream), CRS_EPSG)) +
  ggplot2::theme_minimal(base_size = 9)
ggplot2::ggsave("figures/01b_tile_plan.png", p, width = 7, height = 7, dpi = 200, bg = "white")
p

## 5 · Download links
Training tiles first. Each link returns one tile as a zip from **Cascades North 2023**. If a link returns an empty zip or a different project folder, that tile is outside Cascades North (likely the Acme Valley 2024 strip) — note the tile name and tell Claude.

Save the zips into one folder (keep the default names); where they go next (Drive vs upload) is decided before 01 scales.

In [ ]:
todo <- out[!out$downloaded, ]
links <- sprintf('<li><a href="%s" target="_blank">%03d · %s</a> <small>(%s)</small></li>', todo$url, todo$n, todo$tile, todo$role)
IRdisplay::display_html(paste0("<ol style='columns:3;font-family:monospace;font-size:12px'>", paste(links, collapse = ""), "</ol>"))